# PharmaCopilot — ML Training Notebook

**Purpose:** Train the XGBoost batch failure prediction model and export `pharma_batch_model.pkl`  
**Output:** `pharma_batch_model.pkl` + `pharma_model_meta.json` → PUT to Native App package stage  

## ⚠️ Dev Training vs Native App Install

| Step | Where it runs | Notes |
|------|--------------|-------|
| Cells 1–7 | Dev Snowflake account (`<DEV_DB>` DB) | Training only — reads from dev tables |
| Cell 8 | Local temp dir | Serialises model to `.pkl` |
| Cell 9 | Dev stage | Optional — smoke test only |
| **Cell 10** | **SnowSQL / CLI** | **PUT pkl to Native App package stage** |
| Cell 11 | Dev account | Smoke test against dev UDF |

The Native App UDF (`ML_CODE.PREDICT_BATCH_FAILURE_PROB`) is created by `setup.sql` at install time.  
Do **not** register the UDF manually from this notebook against a consumer app.


## Step 1 — Imports


In [ ]:
import os, json, tempfile, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
import joblib
import sklearn

warnings.filterwarnings('ignore')
print(f'scikit-learn version: {sklearn.__version__}')

# The model MUST be serialised with scikit-learn == 1.3.0 to match the
# pinned PACKAGES version in setup.sql:
#   PACKAGES = ('scikit-learn==1.3.0', ...)
# If this assertion fails: pip install scikit-learn==1.3.0
assert sklearn.__version__.startswith('1.3'), (
    f'Version mismatch: got {sklearn.__version__}, need 1.3.x. '
    'Run: pip install scikit-learn==1.3.0'
)
print('Version check passed.')


## Step 2 — Snowflake Session


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# All training queries run against the DEV database.
# This is intentional — notebook is a dev training script, not a Native App component.
DEV_DB = '<DEV_DB>'
print(f'Connected. Training will read from {DEV_DB}.')


## Step 3 — Load Training Data


In [ ]:
TRAINING_SQL = f"""
SELECT
    mi.PRODUCT_TYPE,     mi.PLANT_ID,
    mi.BATCH_SIZE_BUCKET, mi.DURATION_BUCKET,
    mi.TOTAL_TESTS,      mi.FAILED_TESTS,     mi.OOS_COUNT,       mi.BORDERLINE_COUNT,
    mi.PASS_RATE_PCT,    mi.OOS_RATE_PCT,     mi.FAIL_RATE_PCT,
    mi.STERILITY_FAIL_FLAG, mi.ENDOTOXIN_FAIL_FLAG, mi.STERILITY_X_ENDOTOXIN,
    mi.TEMP_VIOLATION_COUNT, mi.TEMP_MAX_EXCURSION_MINS, mi.AVG_TEMP_DEVIATION_C,
    mi.HUMIDITY_VIOLATION_COUNT, mi.PRESSURE_VIOLATION_COUNT,
    mi.TOTAL_IOT_VIOLATIONS, mi.IOT_VIOLATION_DENSITY, mi.TEMP_X_DURATION,
    mi.TOTAL_DEVIATIONS, mi.CRITICAL_DEVIATION_COUNT, mi.HIGH_DEVIATION_COUNT,
    mi.WEIGHTED_DEVIATION_SCORE, mi.PROCESS_DEVIATION_COUNT,
    mi.EQUIPMENT_DEVIATION_COUNT, mi.HUMAN_DEVIATION_COUNT,
    mi.DEVIATION_DENSITY, mi.CRITICAL_DEV_RATE, mi.TEMP_X_CRITICAL_DEV,
    mi.BATCH_SIZE, mi.BATCH_DURATION_HOURS, mi.PROCESS_VARIANCE,
    mi.BATCH_START_HOUR, mi.BATCH_START_DOW, mi.IS_WEEKEND_BATCH, mi.IS_NIGHT_SHIFT,
    CASE WHEN sb.BATCH_STATUS = 'RELEASED' THEN 0 ELSE 1 END AS TARGET_BINARY,
    sb.BATCH_STATUS AS TARGET_LABEL
FROM {{DEV_DB}}.FEATURE_TEST.FEAT_ML_INPUT mi
JOIN {{DEV_DB}}.SILVER_TEST.HUB_BATCH  hb ON mi.BATCH_ID  = hb.BATCH_ID
JOIN {{DEV_DB}}.SILVER_TEST.SAT_BATCH  sb ON hb.HK_BATCH  = sb.HK_BATCH
"""

df = session.sql(TRAINING_SQL).to_pandas()
print(f'Rows loaded: {len(df):,}')
print(f'Class distribution:\n{df["TARGET_LABEL"].value_counts()}')


## Step 4 — Feature Definition

> **39 features total** (4 categorical + 35 numeric).  
> `BATCH_STATUS` is accepted by the UDF signature for SQL convenience but is **not** a model input feature.


In [ ]:
CAT_COLS = ['PRODUCT_TYPE', 'PLANT_ID', 'BATCH_SIZE_BUCKET', 'DURATION_BUCKET']

NUM_COLS = [
    'TOTAL_TESTS', 'FAILED_TESTS', 'OOS_COUNT', 'BORDERLINE_COUNT',
    'PASS_RATE_PCT', 'OOS_RATE_PCT', 'FAIL_RATE_PCT',
    'STERILITY_FAIL_FLAG', 'ENDOTOXIN_FAIL_FLAG', 'STERILITY_X_ENDOTOXIN',
    'TEMP_VIOLATION_COUNT', 'TEMP_MAX_EXCURSION_MINS', 'AVG_TEMP_DEVIATION_C',
    'HUMIDITY_VIOLATION_COUNT', 'PRESSURE_VIOLATION_COUNT',
    'TOTAL_IOT_VIOLATIONS', 'IOT_VIOLATION_DENSITY', 'TEMP_X_DURATION',
    'TOTAL_DEVIATIONS', 'CRITICAL_DEVIATION_COUNT', 'HIGH_DEVIATION_COUNT',
    'WEIGHTED_DEVIATION_SCORE', 'PROCESS_DEVIATION_COUNT',
    'EQUIPMENT_DEVIATION_COUNT', 'HUMAN_DEVIATION_COUNT',
    'DEVIATION_DENSITY', 'CRITICAL_DEV_RATE', 'TEMP_X_CRITICAL_DEV',
    'BATCH_SIZE', 'BATCH_DURATION_HOURS', 'PROCESS_VARIANCE',
    'BATCH_START_HOUR', 'BATCH_START_DOW', 'IS_WEEKEND_BATCH', 'IS_NIGHT_SHIFT',
]

TARGET_COL = 'TARGET_BINARY'

X = df[CAT_COLS + NUM_COLS].copy()
y = df[TARGET_COL].astype(int)

total_features = len(CAT_COLS) + len(NUM_COLS)
print(f'Features: {len(CAT_COLS)} categorical + {len(NUM_COLS)} numeric = {total_features} total')
assert total_features == 39, f'Expected 39 features, got {total_features}'


## Step 5 — Build Pipeline


In [ ]:
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='UNKNOWN')),
    ('encoder', OrdinalEncoder(
        handle_unknown='use_encoded_value', unknown_value=-1,
        encoded_missing_value=-1
    )),
])

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])

preprocessor = ColumnTransformer([
    ('cat', cat_pipeline, CAT_COLS),
    ('num', num_pipeline, NUM_COLS),
], remainder='drop')

neg_count = (y == 0).sum()
pos_count = (y == 1).sum()
spw = float(neg_count) / max(float(pos_count), 1)
print(f'scale_pos_weight: {spw:.4f}  (neg={neg_count}, pos={pos_count})')

xgb_model = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.80,
    colsample_bytree=0.80,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.05,
    reg_lambda=1.0,
    scale_pos_weight=spw,
    objective='binary:logistic',
    eval_metric='auc',
    use_label_encoder=False,
    random_state=42,
    n_jobs=-1,
)

full_pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', xgb_model),
])
print('Pipeline built.')


## Step 6 — Train & Evaluate


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f'Training on {len(X_train):,} rows, testing on {len(X_test):,} rows')
full_pipeline.fit(X_train, y_train)

y_pred      = full_pipeline.predict(X_test)
y_pred_prob = full_pipeline.predict_proba(X_test)[:, 1]

print('\n--- Classification Report ---')
print(classification_report(y_test, y_pred, target_names=['PASS', 'FAIL']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_pred_prob):.4f}')

cv_auc = cross_val_score(
    full_pipeline, X, y,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc', n_jobs=-1
)
print(f'5-Fold CV AUC: {cv_auc.mean():.4f} +/- {cv_auc.std():.4f}')


## Step 7 — Feature Importances


In [ ]:
feat_names = CAT_COLS + NUM_COLS
raw_imp = full_pipeline.named_steps['model'].feature_importances_
top15 = sorted(zip(feat_names, raw_imp), key=lambda x: -x[1])[:15]
print('Top-15 Feature Importances:')
for name, imp in top15:
    bar = '#' * int(imp * 200)
    print(f'  {name:<40} {imp:.4f}  {bar}')


## Step 8 — Serialise Artifacts

Saves `pharma_batch_model.pkl` and `pharma_model_meta.json` to a local temp directory.


In [ ]:
TMPDIR        = tempfile.mkdtemp()
MODEL_PATH    = os.path.join(TMPDIR, 'pharma_batch_model.pkl')
METADATA_PATH = os.path.join(TMPDIR, 'pharma_model_meta.json')

metadata = {
    'cat_cols':        CAT_COLS,
    'num_cols':        NUM_COLS,
    'all_cols':        CAT_COLS + NUM_COLS,
    'target':          TARGET_COL,
    'roc_auc':         round(float(roc_auc_score(y_test, y_pred_prob)), 4),
    'cv_auc_mean':     round(float(cv_auc.mean()), 4),
    'cv_auc_std':      round(float(cv_auc.std()), 4),
    'spw':             round(spw, 4),
    'sklearn_version': sklearn.__version__,
    'top15_features':  [{'name': n, 'importance': round(float(i), 4)} for n, i in top15],
}

joblib.dump(full_pipeline, MODEL_PATH)
with open(METADATA_PATH, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'Model saved   : {MODEL_PATH}')
print(f'Metadata saved: {METADATA_PATH}')
print(f'sklearn in artifact: {sklearn.__version__}')


## Step 9 — (Optional) Upload to Dev Stage

> Uploads to the **dev** `<DEV_DB>` stage only.  
> Needed only to run the smoke test in Step 11. Skip if you don't need the dev UDF.


In [ ]:
# Optional — skip if you only need the pkl for Native App packaging
DEV_STAGE = f'@{DEV_DB}.FEATURE_TEST.PHARMA_MODEL_STAGE'
print(f'Uploading to dev stage: {DEV_STAGE} ...')
session.file.put(MODEL_PATH,    DEV_STAGE, auto_compress=False, overwrite=True)
session.file.put(METADATA_PATH, DEV_STAGE, auto_compress=False, overwrite=True)
print('Dev stage upload complete.')


## Step 10 — PUT Artifacts to Native App Package Stage ⬅️ Critical

Run these commands in **SnowSQL** or the **Snowflake CLI** after completing Step 8.  
Replace `<PKG_STAGE>` with your actual package stage (e.g. `<PKG_STAGE>`).

```sql
PUT file:///path/to/pharma_batch_model.pkl
    @<PKG_STAGE>/v1/artifacts/pharma_batch_model.pkl
    OVERWRITE=TRUE AUTO_COMPRESS=FALSE;

PUT file:///path/to/pharma_model_meta.json
    @<PKG_STAGE>/v1/artifacts/pharma_model_meta.json
    OVERWRITE=TRUE AUTO_COMPRESS=FALSE;
```

> The path `artifacts/pharma_batch_model.pkl` must match the `IMPORTS` clause in `setup.sql`:  
> `IMPORTS = ('/artifacts/pharma_batch_model.pkl')`

After uploading, add a new patch and reinstall the test app:

```sql
ALTER APPLICATION PACKAGE <APP_PACKAGE_NAME>
    ADD PATCH FOR VERSION V1 USING '@<PKG_STAGE>/v1';

DROP APPLICATION <APP_NAME>;

CREATE APPLICATION <APP_NAME>
    FROM APPLICATION PACKAGE <APP_PACKAGE_NAME>
    USING VERSION V1 PATCH <new_patch_number>;
```


In [ ]:
# Print exact PUT commands with your local paths filled in
print('Run these in SnowSQL (replace <PKG_STAGE> with your stage name):\n')
print(f'PUT file://{MODEL_PATH}')
print(f'    @<PKG_STAGE>/v1/artifacts/pharma_batch_model.pkl')
print(f'    OVERWRITE=TRUE AUTO_COMPRESS=FALSE;\n')
print(f'PUT file://{METADATA_PATH}')
print(f'    @<PKG_STAGE>/v1/artifacts/pharma_model_meta.json')
print(f'    OVERWRITE=TRUE AUTO_COMPRESS=FALSE;')


## Step 11 — Smoke Test (Dev UDF)

> Tests the **dev** UDF at `FEATURE_TEST.PREDICT_BATCH_FAILURE_PROB`.  
> The Native App UDF (`ML_CODE.PREDICT_BATCH_FAILURE_PROB`) is validated by `SP_POST_INSTALL_SETUP()`.


In [ ]:
smoke = session.sql(f"""
    SELECT {DEV_DB}.FEATURE_TEST.PREDICT_BATCH_FAILURE_PROB(
        'Oral Solid', 'PLT-MUMBAI', 'QA_REVIEW', 'LARGE', 'NORMAL',
        25, 3, 1, 2,  88.0, 4.0, 12.0,  0, 0, 0,
        2, 10.0, 1.5,  1, 0,  3, 0.125, 48.0,
        4, 1, 2,  7.0, 1, 1, 1,  0.0008, 0.25, 2,
        5000.0, 24.0, 0.08,
        8, 1, 0, 0
    ) AS SMOKE_PROB
""").to_pandas()

prob = smoke['SMOKE_PROB'].iloc[0]
print(f'Smoke test failure probability: {prob:.4f}')
assert 0.0 <= prob <= 1.0, f'UDF returned out-of-range value: {prob}'
print('Smoke test PASSED. Model is ready for Native App packaging.')
